# 02 Perceptual Training (Including Long Training Runs)

In [ ]:
import os
from PIL import Image
import numpy as np
import tensorflow as tf
from functools import partial
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger

print("TensorFlow version:", tf.__version__)

In [ ]:
train_low_dir = "../data/train/low"
train_high_dir = "../data/train/high"

def load_image(image_path, size=(256, 256)):
    img = Image.open(image_path).convert('RGB')
    img = img.resize(size)
    img = np.array(img).astype(np.float32) / 255.0
    return img

low_light_images, enhanced_images = [], []
for img_name in sorted(os.listdir(train_low_dir)):
    low_path = os.path.join(train_low_dir, img_name)
    high_path = os.path.join(train_high_dir, img_name)
    if os.path.exists(high_path):
        low_light_images.append(load_image(low_path))
        enhanced_images.append(load_image(high_path))

low_light_array = np.array(low_light_images, dtype=np.float32)
enhanced_array = np.array(enhanced_images, dtype=np.float32)
print(f"Loaded training data: {low_light_array.shape}, {enhanced_array.shape}")

In [ ]:
def unet_model(input_shape=(256, 256, 3)):
    inputs = layers.Input(input_shape)
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(p1)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    b = layers.Conv2D(256, 3, activation='relu', padding='same')(p2)
    u2 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(b)
    concat2 = layers.concatenate([u2, c2])
    c3 = layers.Conv2D(128, 3, activation='relu', padding='same')(concat2)
    u1 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(c3)
    concat1 = layers.concatenate([u1, c1])
    c4 = layers.Conv2D(64, 3, activation='relu', padding='same')(concat1)
    outputs = layers.Conv2D(3, 1, activation='sigmoid')(c4)
    return Model(inputs, outputs)

model = unet_model()

In [ ]:
feature_layer_names = ['block2_conv2', 'block3_conv4']
base_vgg = VGG19(include_top=False, weights='imagenet')
base_vgg.trainable = False
feat_outputs = [base_vgg.get_layer(name).output for name in feature_layer_names]
vgg_feat_extractor = Model(inputs=base_vgg.input, outputs=feat_outputs)
vgg_feat_extractor.trainable = False

@tf.function
def perceptual_loss(y_true, y_pred):
    y_true = tf.clip_by_value(y_true, 0.0, 1.0)
    y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)
    y_true_vgg = vgg_preprocess(y_true * 255.0)
    y_pred_vgg = vgg_preprocess(y_pred * 255.0)
    feats_true = vgg_feat_extractor(y_true_vgg)
    feats_pred = vgg_feat_extractor(y_pred_vgg)
    if not isinstance(feats_true, (list, tuple)):
        feats_true = [feats_true]; feats_pred = [feats_pred]
    loss = 0.0
    for ft, fp in zip(feats_true, feats_pred):
        loss += tf.reduce_mean(tf.square(ft - fp))
    loss = loss / float(len(feats_true))
    return loss

In [ ]:
def psnr_metric(y_true, y_pred):
    y_true = tf.clip_by_value(y_true, 0.0, 1.0)
    y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)
    return tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val=1.0))

def ssim_metric(y_true, y_pred):
    y_true = tf.clip_by_value(y_true, 0.0, 1.0)
    y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

def color_alignment_loss(y_true, y_pred):
    mean_true = tf.reduce_mean(y_true, axis=[1,2])
    mean_pred = tf.reduce_mean(y_pred, axis=[1,2])
    std_true = tf.math.reduce_std(y_true, axis=[1,2])
    std_pred = tf.math.reduce_std(y_pred, axis=[1,2])
    return tf.reduce_mean(tf.square(mean_true - mean_pred)) + tf.reduce_mean(tf.square(std_true - std_pred))

def combined_loss(y_true, y_pred):
    mse_loss = tf.reduce_mean(tf.square(y_true - y_pred))
    cal_loss = color_alignment_loss(y_true, y_pred)
    p_loss = perceptual_loss(y_true, y_pred)
    return mse_loss + 0.01 * cal_loss + 0.00001 * p_loss

In [ ]:
save_dir = "../models"
os.makedirs(save_dir, exist_ok=True)
checkpoint_path = os.path.join(save_dir, "best_model_perceptual.keras")
log_path = os.path.join(save_dir, "training_log.csv")

checkpoint_cb = ModelCheckpoint(filepath=checkpoint_path, monitor='loss', save_best_only=True, mode='min', verbose=1)
csv_logger = CSVLogger(log_path, append=True)

model.compile(optimizer='adam', loss=combined_loss, metrics=['mse', psnr_metric, ssim_metric])
history = model.fit(low_light_array, enhanced_array, epochs=20, batch_size=4, callbacks=[checkpoint_cb, csv_logger], verbose=1)

In [ ]:
# Resume training for 100 epochs
from tensorflow.keras.models import load_model
model = load_model(checkpoint_path, compile=False)
model.compile(optimizer='adam', loss=combined_loss, metrics=['mse', psnr_metric, ssim_metric])
history2 = model.fit(low_light_array, enhanced_array, epochs=100, batch_size=4, callbacks=[checkpoint_cb, csv_logger], verbose=1)

In [ ]:
# Fresh 500-epoch training
model = load_model(checkpoint_path, compile=False)
for layer in model.layers:
    if hasattr(layer, 'kernel_initializer'):
        new_weights = [init(w.shape, dtype=w.dtype) for w, init in zip(layer.get_weights(), [layer.kernel_initializer if hasattr(layer, 'kernel_initializer') else None for _ in layer.get_weights()])]
        if new_weights:
            layer.set_weights(new_weights)

model.compile(optimizer='adam', loss=combined_loss, metrics=['mse', psnr_metric, ssim_metric])
run_dir = os.path.join(save_dir, "fresh_training_500")
os.makedirs(run_dir, exist_ok=True)
checkpoint_path_500 = os.path.join(run_dir, "best_model_fresh_500.keras")
checkpoint_cb = ModelCheckpoint(checkpoint_path_500, monitor='val_loss', save_best_only=True, verbose=1)
csv_logger = CSVLogger(os.path.join(run_dir, "training_log_500.csv"), append=False)

history_500 = model.fit(low_light_array, enhanced_array, epochs=500, batch_size=4, validation_split=0.1, callbacks=[checkpoint_cb, csv_logger], verbose=1)

In [ ]:
# Fresh 5000-epoch training
model = load_model(checkpoint_path, compile=False)
for layer in model.layers:
    if hasattr(layer, 'kernel_initializer'):
        new_weights = [init(w.shape, dtype=w.dtype) for w, init in zip(layer.get_weights(), [layer.kernel_initializer if hasattr(layer, 'kernel_initializer') else None for _ in layer.get_weights()])]
        if new_weights:
            layer.set_weights(new_weights)

model.compile(optimizer='adam', loss=combined_loss, metrics=['mse', psnr_metric, ssim_metric])
run_dir = os.path.join(save_dir, "fresh_training_5000")
os.makedirs(run_dir, exist_ok=True)
checkpoint_path_5000 = os.path.join(run_dir, "best_model_fresh_5000.keras")
checkpoint_cb = ModelCheckpoint(checkpoint_path_5000, monitor='val_loss', save_best_only=True, verbose=1)
csv_logger = CSVLogger(os.path.join(run_dir, "training_log_5000.csv"), append=False)

history_5000 = model.fit(low_light_array, enhanced_array, epochs=5000, batch_size=4, validation_split=0.25, callbacks=[checkpoint_cb, csv_logger], verbose=1)